# 🎨 Anime Image Generator — Kaggle + S3
### Model: dreamlike-art/dreamlike-anime-1.0

**Filename format:** `6f_0001_scene_001.png` → `PREFIX_CHAPTER_scene_NNN.png`

**Supported JSON naming:**
- **Scenario A (current):** `f_0001_image_prompts.json`, `f_0002_image_prompts.json` ...
- **Scenario B (old):** `g_chapter_0411.json`, `g_chapter_0412.json` ...

**Before running:**
1. Enable **GPU** → Settings → Accelerator → GPU T4 x2
2. Add secrets → Add-ons → Secrets: `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_REGION`, `S3_BUCKET_NAME`
3. Run cells top to bottom

## ⚙️ Cell 1 — Install Libraries

In [1]:
!pip install diffusers transformers accelerate safetensors xformers boto3 --quiet
print('✅ All libraries installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 35.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 98.8 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompat

## 🔑 Cell 2 — AWS Setup & S3 Connection

In [2]:
import boto3
from botocore.exceptions import ClientError
from google.colab import userdata

# ── Load secrets from Colab Secrets ──
AWS_ACCESS_KEY_ID     = userdata.get('AWS_ACCESS_KEY_ID').strip()
AWS_SECRET_ACCESS_KEY = userdata.get('AWS_SECRET_ACCESS_KEY').strip()
AWS_REGION            = userdata.get('AWS_REGION').strip()
S3_BUCKET_NAME        = userdata.get('S3_BUCKET_NAME').strip()

# ── Multi-chapter S3 paths — keyed by chapter number ──
CHAPTER_PATHS = {
    '02': ('New_Novel/02/json/', 'New_Novel/02/IMG/'),
    '03': ('New_Novel/03/json/', 'New_Novel/03/IMG/'),
    '04': ('New_Novel/04/json/', 'New_Novel/04/IMG/'),
    '05': ('New_Novel/05/json/', 'New_Novel/05/IMG/'),
    '07': ('New_Novel/07/json/', 'New_Novel/07/IMG/'),
    '08': ('New_Novel/08/json/', 'New_Novel/08/IMG/'),
    '09': ('New_Novel/09/json/', 'New_Novel/09/IMG/'),
    '10': ('New_Novel/10/json/', 'New_Novel/10/IMG/'),
    '11': ('New_Novel/11/json/', 'New_Novel/11/IMG/'),
    '12': ('New_Novel/12/json/', 'New_Novel/12/IMG/'),
    '13': ('New_Novel/13/json/', 'New_Novel/13/IMG/'),
}

# ── Set chapter number here ──
ACTIVE_CHAPTER = '02'  # change to: 02 03 04 05 07 08 09 10 11 12 13

S3_JSON_PREFIX, S3_OUTPUT_PREFIX = CHAPTER_PATHS[ACTIVE_CHAPTER]

# ── Create S3 client ──
s3_client = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION,
)

# ── Validate bucket ──
try:
    s3_client.head_bucket(Bucket=S3_BUCKET_NAME)
    print(f'✅ AWS S3 client ready — bucket "{S3_BUCKET_NAME}" accessible')
except ClientError as e:
    code = e.response['Error']['Code']
    raise RuntimeError(f'❌ Bucket error ({code}): {S3_BUCKET_NAME}')

print(f'   Active chapter     : {ACTIVE_CHAPTER}')
print(f'   Reading JSONs from : s3://{S3_BUCKET_NAME}/{S3_JSON_PREFIX}')
print(f'   Uploading images to: s3://{S3_BUCKET_NAME}/{S3_OUTPUT_PREFIX}')

✅ AWS S3 client ready — bucket "mynovels-iqranaeem-247601" accessible
   Reading JSONs from : s3://mynovels-iqranaeem-247601/AWS-upload/B/P_11/Json/
   Uploading images to: s3://mynovels-iqranaeem-247601/AWS-upload/B/P_11/IMG/


## 🗂️ Cell 3 — Config: Prefix & JSON Format

In [3]:
# ─────────────────────────────────────────────────────────────────
#  CHANGE THESE TWO SETTINGS WHEN SWITCHING NOVEL PROJECT
# ─────────────────────────────────────────────────────────────────

# Image filename prefix — e.g. '6f', 'G2', 'H1'
IMAGE_PREFIX = 'B2'

# JSON naming format:
#   'new'  → f_0001_image_prompts.json  (current format)
#   'old'  → g_chapter_0411.json        (old chapter format)
JSON_FORMAT = 'new'

# ─────────────────────────────────────────────────────────────────

if JSON_FORMAT == 'new':
    print(f'✅ Image prefix : {IMAGE_PREFIX}')
    print(f'   JSON format  : NEW  (f_0001_image_prompts.json)')
    print(f'   Example out  : {IMAGE_PREFIX}_0001_scene_001.png')
else:
    print(f'✅ Image prefix : {IMAGE_PREFIX}')
    print(f'   JSON format  : OLD  (g_chapter_0411.json)')
    print(f'   Example out  : {IMAGE_PREFIX}_0411_scene_001.png')

✅ Image prefix : B2
   JSON format  : NEW  (f_0001_image_prompts.json)
   Example out  : B2_0001_scene_001.png


## 📋 Cell 4 — Load All JSON Files from S3

In [4]:
import json
import re

# ── S3 helpers ──
def list_s3_keys(prefix):
    keys = []
    paginator = s3_client.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=S3_BUCKET_NAME, Prefix=prefix):
        for obj in page.get('Contents', []):
            keys.append(obj['Key'])
    return keys

def load_json_from_s3(key):
    response = s3_client.get_object(Bucket=S3_BUCKET_NAME, Key=key)
    content  = response['Body'].read().decode('utf-8')
    return json.loads(content)

def get_existing_images():
    """Return set of filenames already uploaded to S3 output folder."""
    keys = list_s3_keys(S3_OUTPUT_PREFIX)
    return set(k.split('/')[-1] for k in keys if k.endswith('.png'))

# ── Filename builder — supports BOTH JSON naming formats ──
def build_filename(scene):
    """
    Scenario A (JSON_FORMAT='new'):
        source file : f_0001_image_prompts
        output      : 6f_0001_scene_001.png

    Scenario B (JSON_FORMAT='old'):
        source file : f_chapter_0411
        output      : 6f_0411_scene_001.png
    """
    src = scene.get('_source_file', 'unknown')
    num = scene.get('scene', 1)

    if JSON_FORMAT == 'new':
        # matches 4-digit number between underscores: f_0001_image_prompts → 0001
        match = re.search(r'_(\d{4})_', src)
        chap  = match.group(1) if match else '0000'
    else:
        # matches digits after 'chapter_': g_chapter_0411 → 0411
        match = re.search(r'chapter_(\d+)', src)
        chap  = match.group(1) if match else '0000'

    return f'{IMAGE_PREFIX}_{chap}_scene_{num:03d}.png'

# ── Load all JSON files from S3 ──
json_keys = sorted([k for k in list_s3_keys(S3_JSON_PREFIX) if k.endswith('.json')])
print(f'📂 Found {len(json_keys)} JSON file(s) in S3:')
for k in json_keys[:10]:
    print(f'   {k.split("/")[-1]}')
if len(json_keys) > 10:
    print(f'   ... and {len(json_keys)-10} more')

# ── Collect all scenes ──
all_scenes = []
for key in json_keys:
    json_name = key.split('/')[-1].replace('.json', '')
    data      = load_json_from_s3(key)
    scenes    = [data] if isinstance(data, dict) else data
    for scene in scenes:
        if scene.get('parse_ok', True) and scene.get('prompt'):
            scene['_source_file'] = json_name
            all_scenes.append(scene)

# ── Check existing images (skip already generated) ──
existing_images = get_existing_images()
pending = [s for s in all_scenes if build_filename(s) not in existing_images]

print(f'\n✅ Total scenes   : {len(all_scenes)}')
print(f'   Already in S3  : {len(existing_images)} images (will be skipped)')
print(f'   To generate    : {len(pending)}')
print(f'\n📝 Filename preview (first 5):')
for s in all_scenes[:5]:
    fn     = build_filename(s)
    status = '✅ done' if fn in existing_images else '⏳ pending'
    print(f'   {fn}  {status}')

📂 Found 400 JSON file(s) in S3:
   chapter_1331_eng_image_prompts.json
   chapter_1332_eng_image_prompts.json
   chapter_1333_eng_image_prompts.json
   chapter_1334_eng_image_prompts.json
   chapter_1335_eng_image_prompts.json
   chapter_1336_eng_image_prompts.json
   chapter_1337_eng_image_prompts.json
   chapter_1338_eng_image_prompts.json
   chapter_1339_eng_image_prompts.json
   chapter_1340_eng_image_prompts.json
   ... and 390 more

✅ Total scenes   : 2903
   Already in S3  : 0 images (will be skipped)
   To generate    : 2903

📝 Filename preview (first 5):
   B2_1331_scene_001.png  ⏳ pending
   B2_1331_scene_002.png  ⏳ pending
   B2_1331_scene_003.png  ⏳ pending
   B2_1331_scene_004.png  ⏳ pending
   B2_1331_scene_005.png  ⏳ pending


## 🤖 Cell 5 — Load Diffusion Model

In [5]:
import torch
from diffusers import StableDiffusionPipeline

MODEL_ID = 'dreamlike-art/dreamlike-anime-1.0'

print(f'⏳ Loading model: {MODEL_ID}')
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    safety_checker=None,
)
pipe = pipe.to('cuda')

# Use these instead of xformers:
pipe.enable_attention_slicing()          # saves VRAM
pipe.unet.enable_gradient_checkpointing()  # optional, extra safety

print('✅ Model loaded and ready on GPU')

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


⏳ Loading model: dreamlike-art/dreamlike-anime-1.0


model_index.json:   0%|          | 0.00/511 [00:00<?, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--dreamlike-art--dreamlike-anime-1.0/snapshots/dd5f188f512fc922786abf5efcec25d5f807dd1e/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded and ready on GPU


## 🖼️ Cell 6 — Generate & Upload Images

In [ ]:
import io
import time
import torch

# ── Generation settings ──
NEG_PROMPT = (
    "worst quality, low quality, normal quality, blurry, "
    "bad anatomy, bad hands, extra fingers, missing fingers, "
    "deformed face, ugly face, poorly drawn face, "
    "text, watermark, logo, signature, username, "
    "flat shading, simple background, cropped"
)

QUALITY_PREFIX = (
    "masterpiece, best quality, highly detailed anime illustration, "
    "cinematic composition, detailed face, expressive eyes, "
    "detailed hair strands, soft lighting, volumetric lighting, "
    "beautiful background, professional anime movie frame, "
    "ultra detailed environment, sharp focus, high quality shading"
)

WIDTH  = 768
HEIGHT = 432
GUID_SCALE  = 7.5
NUM_STEPS   = 30
SEED        = 42
BATCH_SIZE  = 4  # lower to 2 if OOM

total     = len(pending)
generated = 0
failed    = 0

print(f'🚀 Starting generation — {total} images | batch size: {BATCH_SIZE}')
print(f'   Size: {WIDTH}x{HEIGHT} | Steps: {NUM_STEPS} | CFG: {GUID_SCALE}\n')

for i in range(0, total, BATCH_SIZE):
    batch     = pending[i : i + BATCH_SIZE]
    filenames = [build_filename(s) for s in batch]

    # filter already done inside batch
    todo_idx  = [j for j, fn in enumerate(filenames) if fn not in existing_images]
    if not todo_idx:
        print(f'   [batch {i//BATCH_SIZE+1}] ⏭️ All already in S3 — skipping')
        continue

    batch     = [batch[j]     for j in todo_idx]
    filenames = [filenames[j] for j in todo_idx]

    prompts   = [f"{QUALITY_PREFIX}, {s.get('prompt','').strip()}" for s in batch]
    neg_batch = [NEG_PROMPT] * len(batch)
    generators = [torch.Generator('cuda').manual_seed(SEED + i + j) for j in range(len(batch))]

    try:
        t0 = time.time()

        images = pipe(
            prompts,
            negative_prompt=neg_batch,
            width=WIDTH,
            height=HEIGHT,
            guidance_scale=GUID_SCALE,
            num_inference_steps=NUM_STEPS,
            generator=generators,
        ).images

        for fn, img in zip(filenames, images):
            buf = io.BytesIO()
            img.save(buf, format='PNG')
            buf.seek(0)
            s3_client.put_object(
                Bucket=S3_BUCKET_NAME,
                Key=S3_OUTPUT_PREFIX + fn,
                Body=buf,
                ContentType='image/png',
            )
            generated += 1

        elapsed = time.time() - t0
        print(f'   [batch {i//BATCH_SIZE+1}] ✅ {filenames} — {elapsed:.1f}s')

    except torch.cuda.OutOfMemoryError:
        failed += len(batch)
        print(f'   [batch {i//BATCH_SIZE+1}] ❌ OOM — try BATCH_SIZE=2')
        torch.cuda.empty_cache()

    except Exception as e:
        failed += len(batch)
        print(f'   [batch {i//BATCH_SIZE+1}] ❌ Error: {e}')

print(f'\n🎉 Done! Generated: {generated} | Failed: {failed} | Skipped: {total - generated - failed}')

Token indices sequence length is longer than the specified maximum sequence length for this model (89 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['realistic , 8 k , cinematic composition , high detail , professional color grading <|endoftext|><|endoftext|>', ', photorealistic , 8 k , cinematic composition , high detail , professional color grading', ', 8 k , cinematic composition , high detail , professional color grading <|endoftext|><|endoftext|><|endoftext|>', ', 8 k , cinematic composition , high detail , professional color grading <|endoftext|><|endoftext|><|endoftext|>']


🚀 Starting generation — 2903 images | batch size: 4
   Size: 768x432 | Steps: 30 | CFG: 7.5



  0%|          | 0/30 [00:00<?, ?it/s]

The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: [', 8 k , cinematic composition , high detail , professional color grading <|endoftext|>', ', 8 k , cinematic composition , high detail , professional color grading <|endoftext|>', ', 8 k , cinematic composition , high detail , professional color grading <|endoftext|>', 'realistic , 8 k , cinematic composition , high detail , professional color grading']


   [batch 1] ✅ ['B2_1331_scene_001.png', 'B2_1331_scene_002.png', 'B2_1331_scene_003.png', 'B2_1331_scene_004.png'] — 38.3s


...

   [batch 191] ✅ ['B2_1435_scene_007.png', 'B2_1436_scene_001.png', 'B2_1436_scene_002.png', 'B2_1436_scene_003.png'] — 45.1s


  0%|          | 0/30 [00:00<?, ?it/s]

The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['k , cinematic composition , high detail , professional color grading <|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>', ', 8 k , cinematic composition , high detail , professional color grading <|endoftext|><|endoftext|><|endoftext|>', '8 k , cinematic composition , high detail , professional color grading <|endoftext|><|endoftext|><|endoftext|><|endoftext|>', ', photorealistic , 8 k , cinematic composition , high detail , professional color grading']


   [batch 192] ✅ ['B2_1436_scene_004.png', 'B2_1436_scene_005.png', 'B2_1436_scene_006.png', 'B2_1437_scene_001.png'] — 44.9s


  0%|          | 0/30 [00:00<?, ?it/s]

## 📋 Cell 7 — Verify: List All Images in S3

In [ ]:
uploaded = list_s3_keys(S3_OUTPUT_PREFIX)
images   = sorted([k for k in uploaded if k.endswith('.png')])

print(f'📦 s3://{S3_BUCKET_NAME}/{S3_OUTPUT_PREFIX}')
print(f'   Total images: {len(images)}')
print()
for img in images:
    print(f'   ✅ {img.split("/")[-1]}')

## 🛑 Cell 8 — Stop Session

In [ ]:
import os
print('🛑 Stopping session to save GPU resources...')
os._exit(0)